In [7]:
all_entities = ['Contraindication', 'Eq-Value', 'Severity', 'Drug', 'Observation-Name', 'Age', 'Location', 'Organism-Name', 'Encounter', 'Drug-Name', 'Ethnicity', 'Modifier', 'Condition', 'Eq-Unit', 'Eq-Temporal-Unit', 'Family-Member', 'Organism', 'Other', 'Immunization-Name', 'Polarity', 'Condition-Type', 'Immunization', 'Eq-Operator', 'Eq-Temporal-Recency', 'Procedure-Name', 'Indication', 'Exception', 'Study', 'Language', 'Coreference', 'Provider', 'Acuteness', 'Life-Stage-And-Gender', 'Procedure', 'Risk', 'Death', 'Assertion', 'Allergy-Name', 'Specimen', 'Negation', 'Code', 'Stability', 'Birth', 'Criteria-Count', 'Eq-Comparison', 'Condition-Name', 'Insurance', 'Observation', 'Allergy', 'Eq-Temporal-Period']

In [8]:
import json
from typing import Dict, List, Any
from fhir.resources.patient import Patient
from fhir.resources.procedure import Procedure

def parse_json_to_fhir(json_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Parse the input nested JSON data into FHIR resources.
    
    :param json_data: Input nested JSON data
    :return: List of FHIR resources
    """
    fhir_resources = []

    def traverse_ast(node: Dict[str, Any]) -> Dict[str, Any]:
        if "raw_text" in node:
            return create_fhir_resource(node)

        operator = list(node.keys())[0]
        if operator in ["AND", "OR"]:
            left = traverse_ast(node[operator]["left"])
            right = traverse_ast(node[operator]["right"])
            return merge_resources(left, right, operator)
        elif operator == "NOT":
            return negate_resource(traverse_ast(node[operator]["left"]))
        else:
            # Recursive case for nested structures
            return traverse_ast(node[operator])

    def create_fhir_resource(node: Dict[str, Any]) -> Dict[str, Any]:
        resource_type = determine_resource_type(node)
        resource_data = {}

        for entity, values in node.items():
            if entity != "raw_text":
                resource_data[entity] = values

        try:
            if resource_type == "Patient":
                fhir_resource = Patient(**resource_data).dict()
            elif resource_type == "Procedure":
                fhir_resource = Procedure(**resource_data).dict()
            else:
                # Fallback for other resource types
                fhir_resource = resource_data

            return {"resourceType": resource_type, **fhir_resource}
        except ValueError as e:
            print(f"Error creating FHIR resource: {e}")
            return {}

    def determine_resource_type(node: Dict[str, Any]) -> str:
        if "Age" in node or "Life-Stage-And-Gender" in node:
            return "Patient"
        elif "Procedure" in node or "Procedure-Name" in node:
            return "Procedure"
        else:
            return "Other"

    def merge_resources(left: Dict[str, Any], right: Dict[str, Any], operator: str) -> Dict[str, Any]:
        merged = left.copy()
        for key, value in right.items():
            if key in merged:
                if isinstance(merged[key], list):
                    merged[key].extend(value if isinstance(value, list) else [value])
                else:
                    merged[key] = [merged[key], value] if isinstance(value, list) else [merged[key], value]
            else:
                merged[key] = value
        return merged

    def negate_resource(resource: Dict[str, Any]) -> Dict[str, Any]:
        negated = resource.copy()
        if "modifierExtension" not in negated:
            negated["modifierExtension"] = []
        negated["modifierExtension"].append({
            "url": "http://hl7.org/fhir/StructureDefinition/data-absent-reason",
            "valueCode": "not-applicable"
        })
        return negated

    # Start traversing the AST from the root
    result = traverse_ast(json_data)
    fhir_resources.append(result)

    return fhir_resources

# Read the input JSON file
with open("C:\\Users\\e-aut\\DataspellProjects\\ec_criteria_struct\\lct\\fhir_parser\\data\\input.json", "r", encoding="utf-8") as file:
    json_data = json.load(file)

# Parse the JSON data to FHIR resources
fhir_resources = parse_json_to_fhir(json_data)

# Print the resulting FHIR resources
print(json.dumps(fhir_resources, indent=2))

Error creating FHIR resource: 5 validation errors for Patient
Age
  extra fields not permitted (type=value_error.extra)
Eq-Comparison
  extra fields not permitted (type=value_error.extra)
Eq-Operator
  extra fields not permitted (type=value_error.extra)
Eq-Temporal-Unit
  extra fields not permitted (type=value_error.extra)
Eq-Value
  extra fields not permitted (type=value_error.extra)
Error creating FHIR resource: 2 validation errors for Patient
Eq-Operator
  extra fields not permitted (type=value_error.extra)
Life-Stage-And-Gender
  extra fields not permitted (type=value_error.extra)
Error creating FHIR resource: 1 validation error for Procedure
__root__ -> status
  field required (type=value_error.missing)
Error creating FHIR resource: 1 validation error for Procedure
__root__ -> status
  field required (type=value_error.missing)
Error creating FHIR resource: 1 validation error for Procedure
__root__ -> status
  field required (type=value_error.missing)
[
  {
    "resourceType": [
  

In [9]:
import json
from typing import Dict, List, Any
from fhir.resources.patient import Patient
from fhir.resources.procedure import Procedure
from fhir.resources.basic import Basic
from fhir.resources.extension import Extension
from fhir.resources.codeableconcept import CodeableConcept

# Liste der bekannten Entitäten
all_entities = [
    'Contraindication', 'Eq-Value', 'Severity', 'Drug', 'Observation-Name', 'Age', 'Location', 'Organism-Name',
    'Encounter', 'Drug-Name', 'Ethnicity', 'Modifier', 'Condition', 'Eq-Unit', 'Eq-Temporal-Unit', 'Family-Member',
    'Organism', 'Other', 'Immunization-Name', 'Polarity', 'Condition-Type', 'Immunization', 'Eq-Operator',
    'Eq-Temporal-Recency', 'Procedure-Name', 'Indication', 'Exception', 'Study', 'Language', 'Coreference',
    'Provider', 'Acuteness', 'Life-Stage-And-Gender', 'Procedure', 'Risk', 'Death', 'Assertion', 'Allergy-Name',
    'Specimen', 'Negation', 'Code', 'Stability', 'Birth', 'Criteria-Count', 'Eq-Comparison', 'Condition-Name',
    'Insurance', 'Observation', 'Allergy', 'Eq-Temporal-Period'
]

def parse_json_to_fhir(json_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Parse the input nested JSON data into FHIR resources.
    
    :param json_data: Input nested JSON data
    :return: List of FHIR resources
    """
    fhir_resources = []

    def traverse_ast(node: Dict[str, Any]) -> Dict[str, Any]:
        if "raw_text" in node:
            return create_fhir_resource(node)

        operator = list(node.keys())[0]
        if operator in ["AND", "OR"]:
            left = traverse_ast(node[operator]["left"])
            right = traverse_ast(node[operator]["right"])
            return merge_resources(left, right, operator)
        elif operator == "NOT":
            return negate_resource(traverse_ast(node[operator]["left"]))
        else:
            # Recursive case for nested structures
            return traverse_ast(node[operator])

    def create_fhir_resource(node: Dict[str, Any]) -> Dict[str, Any]:
        resource_data = {}
        extensions = []

        for entity, values in node.items():
            if entity != "raw_text":
                if entity in all_entities:
                    # Erstellen einer Erweiterung für jede Entität
                    extension = {
                        "url": f"http://example.org/fhir/StructureDefinition/{entity}",
                        "valueString": ', '.join(map(str, values)) if isinstance(values, list) else str(values)
                    }
                    extensions.append(extension)
                else:
                    resource_data[entity] = values

        # Verwenden der Basic-Ressource als generisches Containerobjekt
        fhir_resource = Basic(
            code=CodeableConcept(text=node.get("raw_text", "Unknown")),
            extension=[Extension(**ext) for ext in extensions]
        )

        return fhir_resource.dict()

    def merge_resources(left: Dict[str, Any], right: Dict[str, Any], operator: str) -> Dict[str, Any]:
        merged = {
            "resourceType": "Basic",
            "extension": []
        }

        # Füge die Informationen aus den beiden Ressourcen hinzu
        merged["extension"].extend(left.get("extension", []))
        merged["extension"].extend(right.get("extension", []))

        # Füge eine Erweiterung hinzu, um den logischen Operator zu repräsentieren
        operator_extension = {
            "url": "http://example.org/fhir/StructureDefinition/operator",
            "valueString": operator
        }
        merged["extension"].append(operator_extension)

        return merged

    def negate_resource(resource: Dict[str, Any]) -> Dict[str, Any]:
        negated = resource.copy()
        if "modifierExtension" not in negated:
            negated["modifierExtension"] = []
        negated["modifierExtension"].append({
            "url": "http://hl7.org/fhir/StructureDefinition/data-absent-reason",
            "valueCode": "not-applicable"
        })

        # Füge eine Erweiterung hinzu, um den NOT-Operator zu repräsentieren
        operator_extension = {
            "url": "http://example.org/fhir/StructureDefinition/operator",
            "valueString": "NOT"
        }
        if "extension" not in negated:
            negated["extension"] = []
        negated["extension"].append(operator_extension)

        return negated

    # Start traversing the AST from the root
    result = traverse_ast(json_data)
    fhir_resources.append(result)

    return fhir_resources

# Read the input JSON file


In [14]:
with open("C:\\Users\\e-aut\\DataspellProjects\\ec_criteria_struct\\lct\\fhir_parser\\data\\input.json", "r", encoding="utf-8") as file:
    json_data = json.load(file)


In [15]:
def create_characteristic(description, exclude=False):
    return {
        "description": description,
        "exclude": exclude
    }

def create_definition_by_combination(code, characteristics):
    return {
        "definitionByCombination": {
            "code": code,
            "characteristic": characteristics
        }
    }

def process_node(node):
    if isinstance(node, dict):
        if "AND" in node:
            left = process_node(node["AND"]["left"])
            right = process_node(node["AND"]["right"])
            return create_definition_by_combination("all-of", [left, right])
        elif "OR" in node:
            left = process_node(node["OR"]["left"])
            right = process_node(node["OR"]["right"])
            return create_definition_by_combination("any-of", [left, right])
        elif "NOT" in node:
            inner = process_node(node["NOT"]["left"])
            inner["exclude"] = True
            return inner
        else:
            description = node.get("raw_text", "")
            characteristic = create_characteristic(description)

            for key, value in node.items():
                if key not in ["raw_text", "Eq-Operator"]:
                    characteristic[key] = value

            return characteristic
    return None

def convert_json_to_fhir(input_json):
    fhir_structure = process_node(input_json)

    return {
        "resourceType": "EvidenceVariable",
        "status": "draft",
        "characteristic": [fhir_structure]
    }
fhir_output = convert_json_to_fhir(json_data)
print(json.dumps(fhir_output, indent=2))

{
  "resourceType": "EvidenceVariable",
  "status": "draft",
  "characteristic": [
    {
      "definitionByCombination": {
        "code": "all-of",
        "characteristic": [
          {
            "definitionByCombination": {
              "code": "all-of",
              "characteristic": [
                {
                  "definitionByCombination": {
                    "code": "all-of",
                    "characteristic": [
                      {
                        "description": "- Age 8yr-18yrs",
                        "exclude": false,
                        "Age": [
                          "Age"
                        ],
                        "Eq-Comparison": [
                          "8yr-18yrs"
                        ],
                        "Eq-Value": [
                          "8",
                          "18"
                        ],
                        "Eq-Temporal-Unit": [
                          "yr",
                          "yrs"